# Financing cash flows of a delta-hedged call

This notebook follows a dealer who **sells one European call** and delta-hedges it with shares. The hedge is self-financing: option premium, stock trades, and interest all pass through one cash account.

Cash-flow signs are from the dealer's perspective: **positive = cash received**, **negative = cash paid**. A negative cash-account balance is borrowing; a positive balance is cash invested. The example ignores dividends and transaction costs so the financing mechanics stay visible.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.options.display.float_format = "{:,.2f}".format


def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_call_price_delta(spot, strike, tau, rate, volatility):
    """Black--Scholes call value and delta, with no dividends."""
    if tau <= 0:
        return max(spot - strike, 0.0), float(spot > strike)
    vol_sqrt_t = volatility * math.sqrt(tau)
    d1 = (math.log(spot / strike) + (rate + 0.5 * volatility**2) * tau) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    price = spot * normal_cdf(d1) - strike * math.exp(-rate * tau) * normal_cdf(d2)
    return price, normal_cdf(d1)

## Simple market path

The call has a strike of 100 and one year to expiry. We rebalance quarterly along one illustrative stock-price path. Change `spots` to see how the financing need changes when the underlying follows a different path.

In [ ]:
strike = 100.0
rate = 0.05
volatility = 0.20
maturity = 1.0
times = np.array([0.00, 0.25, 0.50, 0.75, 1.00])
spots = np.array([100.0, 105.0, 98.0, 110.0, 108.0])

option_premium, initial_delta = bs_call_price_delta(
    spots[0], strike, maturity, rate, volatility
)

print(f"Call premium received: {option_premium:,.2f}")
print(f"Initial call delta:      {initial_delta:,.4f}")
print(f"Initial shares bought:  {initial_delta:,.4f}")
print(f"Initial borrowing:      {initial_delta * spots[0] - option_premium:,.2f}")

## Build the self-financing cash-flow ledger

At each rebalance, the prior cash balance first accrues at the risk-free rate. The stock position is then moved to the new call delta. At expiry, the stock hedge is sold and the call payoff is paid. No outside money is added after inception.

For each interval, `interest = opening cash × (exp(r × Δt) − 1)`. Thus interest is negative when the hedge is financed by borrowing.

In [ ]:
rows = []

# Inception: sell the call and buy delta shares.
shares = initial_delta
stock_trade_cf = -shares * spots[0]
cash = option_premium + stock_trade_cf
rows.append({
    "time": times[0],
    "spot": spots[0],
    "target_delta": shares,
    "opening_cash": 0.0,
    "option_cf": option_premium,
    "interest_cf": 0.0,
    "stock_trade_cf": stock_trade_cf,
    "closing_cash": cash,
    "event": "Sell call + establish hedge",
})

# Intermediate quarterly hedge rebalances.
for i in range(1, len(times) - 1):
    dt = times[i] - times[i - 1]
    opening_cash = cash
    interest_cf = opening_cash * math.expm1(rate * dt)
    _, target_delta = bs_call_price_delta(
        spots[i], strike, maturity - times[i], rate, volatility
    )
    shares_traded = target_delta - shares
    stock_trade_cf = -shares_traded * spots[i]
    cash = opening_cash + interest_cf + stock_trade_cf
    shares = target_delta
    rows.append({
        "time": times[i],
        "spot": spots[i],
        "target_delta": shares,
        "opening_cash": opening_cash,
        "option_cf": 0.0,
        "interest_cf": interest_cf,
        "stock_trade_cf": stock_trade_cf,
        "closing_cash": cash,
        "event": "Rebalance hedge",
    })

# Expiry: accrue the last period, sell the shares, and settle the short call.
dt = times[-1] - times[-2]
opening_cash = cash
interest_cf = opening_cash * math.expm1(rate * dt)
stock_trade_cf = shares * spots[-1]
payoff = max(spots[-1] - strike, 0.0)
option_cf = -payoff
cash = opening_cash + interest_cf + stock_trade_cf + option_cf
rows.append({
    "time": times[-1],
    "spot": spots[-1],
    "target_delta": 0.0,
    "opening_cash": opening_cash,
    "option_cf": option_cf,
    "interest_cf": interest_cf,
    "stock_trade_cf": stock_trade_cf,
    "closing_cash": cash,
    "event": "Unwind hedge + settle call",
})

ledger = pd.DataFrame(rows)
ledger["net_cash_flow"] = (
    ledger["option_cf"] + ledger["interest_cf"] + ledger["stock_trade_cf"]
)
ledger = ledger[[
    "time", "event", "spot", "target_delta", "opening_cash",
    "option_cf", "stock_trade_cf", "interest_cf",
    "net_cash_flow", "closing_cash",
]]

# Accounting check: every closing balance must reconcile exactly.
reconciled = np.allclose(
    ledger["closing_cash"],
    ledger["opening_cash"] + ledger["net_cash_flow"],
)
assert reconciled

display(ledger)

## Financing breakdown and result

`stock_trade_cf` includes the initial hedge, subsequent rebalances, and the final hedge sale. `interest_cf` is the pure financing carry generated by the cash account. The **total financing P&L** is highlighted below. The last closing balance is the terminal P&L of the discretely hedged short call.

In [ ]:
financing_pnl = ledger["interest_cf"].sum()
terminal_hedged_pnl = ledger.iloc[-1]["closing_cash"]

breakdown = pd.Series({
    "Option cash flows (premium - payoff)": ledger["option_cf"].sum(),
    "Stock hedge cash flows": ledger["stock_trade_cf"].sum(),
    "TOTAL FINANCING P&L": financing_pnl,
    "Terminal hedged P&L": terminal_hedged_pnl,
}, name="cash flow")

def highlight_financing_total(row):
    style = "background-color: #FFE699; color: #7F6000; font-weight: bold;"
    return [style if row.name == "TOTAL FINANCING P&L" else "" for _ in row]

display(
    breakdown.to_frame()
    .style.format("{:,.2f}")
    .apply(highlight_financing_total, axis=1)
)

financing_color = "#B91C1C" if financing_pnl < 0 else "#15803D"
display(Markdown(
    f"### Total financing P&L: <span style='color:{financing_color}'>"
    f"**{financing_pnl:,.2f}**</span>"
))
print(f"Final delta-hedged P&L: {terminal_hedged_pnl:,.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ledger["time"], ledger["spot"], marker="o", label="Stock")
axes[0].axhline(strike, color="grey", linestyle="--", label="Strike")
axes[0].set(title="Illustrative stock path", xlabel="Years", ylabel="Price")
axes[0].legend()
axes[0].grid(alpha=0.25)

cash_flow_columns = ["option_cf", "stock_trade_cf", "interest_cf"]
ledger.set_index("time")[cash_flow_columns].plot(
    kind="bar", ax=axes[1], color=["#4C78A8", "#F58518", "#E45756"]
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Dealer cash flows by event", xlabel="Years", ylabel="Cash flow")
axes[1].legend(["Option", "Stock hedge", "Interest"])
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

### How to read the result

- At inception, the premium finances only part of the delta hedge, so the dealer borrows the difference.
- A rise in delta requires more shares and more cash; a fall in delta releases cash.
- Negative cash balances generate negative interest cash flows.
- At expiry, selling the hedge and paying the option payoff closes all positions. The residual is not guaranteed to be zero because hedging is only quarterly and the chosen path need not realize the volatility assumed in the option price.